# Chess.com → PGN → win_rate_by_color

Descarga el historial de partidas de **sugeema** desde la API pública de Chess.com
parseando los PGNs (mismo método que `chess_errors_by_stage`) y genera CSVs
con el win rate desagregado por color (blancas / negras) y modalidad.

In [ ]:
!pip install python-chess requests tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/InfoVis/tp-chess/chess_data"

In [ ]:
import requests, time, os, io
import chess.pgn
import pandas as pd
from tqdm import tqdm

USERNAME = "sugeema"
HEADERS  = {"User-Agent": "Mozilla/5.0 (compatible; ChessDataBot/1.0)"}

In [ ]:
def get_archives(username):
    url = f"https://api.chess.com/pub/player/{username}/games/archives"
    res = requests.get(url, headers=HEADERS)
    if res.status_code != 200:
        print("❌ Error al obtener archivos:", res.status_code)
        return []
    return res.json().get("archives", [])

archives = get_archives(USERNAME)
print(f"📦 Total archivos disponibles: {len(archives)}")
print("Primer archivo:", archives[0]  if archives else "—")
print("Último archivo: ", archives[-1] if archives else "—")

In [ ]:
def download_all_games(username, archives):
    """
    Descarga todos los PGNs y parsea los headers de cada partida.
    Mismo enfoque que chess_errors_by_stage: usa /pgn en lugar de /games.
    """
    records = []

    for archive_url in tqdm(archives, desc="Descargando archivos"):
        res = requests.get(archive_url + "/pgn", headers=HEADERS)

        if res.status_code != 200:
            print(f"⚠️ Error en {archive_url}: {res.status_code}")
            time.sleep(1)
            continue

        pgn_io = io.StringIO(res.text)

        while True:
            game = chess.pgn.read_game(pgn_io)
            if game is None:
                break

            h = game.headers
            white_user = h.get("White", "").lower()
            black_user = h.get("Black", "").lower()

            if white_user == username.lower():
                color      = "white"
                raw_result = h.get("Result", "")
                my_rating  = h.get("WhiteElo", None)
                opp_rating = h.get("BlackElo", None)
                opp_user   = h.get("Black", "")
            elif black_user == username.lower():
                color      = "black"
                raw_result = h.get("Result", "")
                my_rating  = h.get("BlackElo", None)
                opp_rating = h.get("WhiteElo", None)
                opp_user   = h.get("White", "")
            else:
                continue

            # Normalizar resultado según color
            if color == "white":
                if raw_result == "1-0":        outcome = "win"
                elif raw_result == "0-1":      outcome = "loss"
                elif raw_result == "1/2-1/2": outcome = "draw"
                else:                          outcome = "other"
            else:
                if raw_result == "0-1":        outcome = "win"
                elif raw_result == "1-0":      outcome = "loss"
                elif raw_result == "1/2-1/2": outcome = "draw"
                else:                          outcome = "other"

            # Apertura desde ECOUrl (ej: "kings-pawn-opening")
            eco_url  = h.get("ECOUrl", "")
            opening  = eco_url.split("/")[-1] if eco_url else ""

            records.append({
                "date"        : h.get("Date", ""),
                "time_class"  : h.get("TimeClass", ""),
                "color"       : color,
                "outcome"     : outcome,
                "raw_result"  : raw_result,
                "my_rating"   : int(my_rating)  if my_rating  and str(my_rating).isdigit()  else None,
                "opp_rating"  : int(opp_rating) if opp_rating and str(opp_rating).isdigit() else None,
                "opp_username": opp_user,
                "opening"     : opening,
            })

        time.sleep(0.5)  # evitar rate limit

    return records


records = download_all_games(USERNAME, archives)
print(f"\n✅ Partidas descargadas: {len(records)}")

In [ ]:
df_raw = pd.DataFrame(records)

# Parsear fecha
df_raw["date"] = pd.to_datetime(df_raw["date"], format="%Y.%m.%d", errors="coerce")
df_raw["year"] = df_raw["date"].dt.year
df_raw["month"] = df_raw["date"].dt.to_period("M").astype(str)

# Diferencia de rating
df_raw["rating_diff"] = df_raw["my_rating"] - df_raw["opp_rating"]

# Filtrar solo resultados válidos
df = df_raw[df_raw["outcome"].isin(["win", "draw", "loss"])].copy()

print(f"Partidas totales (sin 'other'): {len(df)}")
print()
print(df[["time_class", "color", "outcome"]].value_counts().sort_index())
df.head()

In [ ]:
def win_rate_summary(df, group_cols):
    """
    Calcula win / draw / loss rate agrupado por group_cols.
    Retorna un DataFrame con columnas: win, draw, loss, total, win_pct, draw_pct, loss_pct.
    """
    total  = df.groupby(group_cols).size().rename("total")
    counts = df.groupby(group_cols + ["outcome"]).size().unstack(fill_value=0)

    for col in ["win", "draw", "loss"]:
        if col not in counts.columns:
            counts[col] = 0

    summary = counts[["win", "draw", "loss"]].join(total)
    summary["win_pct"]  = (summary["win"]  / summary["total"] * 100).round(1)
    summary["draw_pct"] = (summary["draw"] / summary["total"] * 100).round(1)
    summary["loss_pct"] = (summary["loss"] / summary["total"] * 100).round(1)

    return summary.reset_index()

In [ ]:
# ── 1. Win rate por COLOR (white / black / both) ──────────────────────────────
wr_color = win_rate_summary(df, ["color"])

df_both = df.copy()
df_both["color"] = "both"
wr_both = win_rate_summary(df_both, ["color"])

wr_color_full = pd.concat([wr_color, wr_both], ignore_index=True)

print("── Win rate por color ──")
print(wr_color_full.to_string(index=False))

In [ ]:
# ── 2. Win rate por COLOR × MODALIDAD ────────────────────────────────────────
wr_color_mode = win_rate_summary(df, ["time_class", "color"])

wr_both_mode  = win_rate_summary(df_both, ["time_class", "color"])
wr_color_mode_full = (
    pd.concat([wr_color_mode, wr_both_mode], ignore_index=True)
    .sort_values(["time_class", "color"])
    .reset_index(drop=True)
)

print("── Win rate por color × modalidad ──")
print(wr_color_mode_full.to_string(index=False))

In [ ]:
# ── 3. Win rate mensual por COLOR (evolución temporal) ────────────────────────
wr_monthly      = win_rate_summary(df, ["month", "color"])
wr_monthly_both = win_rate_summary(df_both, ["month", "color"])
wr_monthly_full = (
    pd.concat([wr_monthly, wr_monthly_both], ignore_index=True)
    .sort_values(["month", "color"])
    .reset_index(drop=True)
)

print("── Win rate mensual (últimas 12 filas) ──")
print(wr_monthly_full.tail(12).to_string(index=False))

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# CSV 1 — partidas crudas (todas las columnas, reutilizable en otros análisis)
raw_path = f"{OUTPUT_DIR}/games_raw.csv"
df.to_csv(raw_path, index=False)

# CSV 2 — win rate por color + fila 'both'  →  gráfico de barras principal
wr_color_path = f"{OUTPUT_DIR}/win_rate_by_color.csv"
wr_color_full.to_csv(wr_color_path, index=False)

# CSV 3 — win rate por color × modalidad  →  filtro por rapid/blitz/daily en Tableau
wr_mode_path = f"{OUTPUT_DIR}/win_rate_by_color_and_mode.csv"
wr_color_mode_full.to_csv(wr_mode_path, index=False)

# CSV 4 — win rate mensual  →  evolución temporal si se quiere agregar
wr_monthly_path = f"{OUTPUT_DIR}/win_rate_monthly.csv"
wr_monthly_full.to_csv(wr_monthly_path, index=False)

print("✅ CSVs guardados en", OUTPUT_DIR)
print(" ", raw_path)
print(" ", wr_color_path)
print(" ", wr_mode_path)
print(" ", wr_monthly_path)

In [ ]:
# Preview del CSV principal que va a Tableau
print("── win_rate_by_color.csv ──")
pd.read_csv(wr_color_path)